# 🍎 CNN vs Transfer Learning — Classification de Fruits (Fruits-360)

**Projet pédagogique** — Comparaison d'un CNN from scratch vs EfficientNetB0 (Transfer Learning) sur le dataset Fruits-360 (141 classes, ~90 000 images).

| | CNN From Scratch | Transfer Learning (EfficientNetB0) |
|---|---|---|
| Paramètres | ~1.2M | ~4.5M |
| Pré-entraîné | ❌ Non | ✅ ImageNet |
| Temps estimé (Colab T4) | ~20 min | ~30 min |

> ⚡ **Recommandé : activer le GPU dans Colab** → Runtime > Change runtime type > T4 GPU

## 1. Installation & Imports

In [ ]:
# Installation des dépendances
!pip install -q opendatasets tensorflow matplotlib seaborn scikit-learn opencv-python-headless

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix

# Vérification GPU
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU disponible:', gpus if gpus else 'Aucun GPU — Activer le GPU dans Colab !')

# Reproductibilité
tf.random.set_seed(42)
np.random.seed(42)

## 2. Téléchargement du Dataset Fruits-360

In [ ]:
import opendatasets as od

# Téléchargement du dataset Fruits-360
# Quand demandé, entrez votre username et API key Kaggle :
#   → Allez sur https://www.kaggle.com/settings
#   → Cliquez "Create New Token" → un fichier kaggle.json est téléchargé
#   → Ouvrez-le et copiez "username" et "key"
od.download("https://www.kaggle.com/datasets/moltean/fruits")

In [ ]:
import glob

# Localisation des dossiers train/test
train_dirs = glob.glob('/content/fruits/**/Training', recursive=True)
test_dirs  = glob.glob('/content/fruits/**/Test',     recursive=True)

TRAIN_DIR = train_dirs[0] if train_dirs else '/content/fruits/fruits-360_dataset_100x100/fruits-360/Training'
TEST_DIR  = test_dirs[0]  if test_dirs  else '/content/fruits/fruits-360_dataset_100x100/fruits-360/Test'

print('Train:', TRAIN_DIR)
print('Test: ', TEST_DIR)

## 3. Exploration du Dataset

In [ ]:
# Statistiques du dataset
classes = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(classes)

total_train = sum(len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in classes)
total_test  = sum(len(os.listdir(os.path.join(TEST_DIR,  c))) for c in classes)

print(f'Nombre de classes  : {NUM_CLASSES}')
print(f'Images train       : {total_train:,}')
print(f'Images test        : {total_test:,}')
print(f'Exemples de classes: {classes[:10]}')

In [ ]:
# Visualisation de quelques images par classe
selected_classes = classes[:12]
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Exemples du dataset Fruits-360', fontsize=16, fontweight='bold')

for ax, cls in zip(axes.flat, selected_classes):
    class_path = os.path.join(TRAIN_DIR, cls)
    img_file = os.listdir(class_path)[0]
    img = plt.imread(os.path.join(class_path, img_file))
    ax.imshow(img)
    ax.set_title(cls, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution des classes (top 20)
class_counts = {c: len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in classes}
top20 = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:20]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar([x[0] for x in top20], [x[1] for x in top20], color='steelblue')
ax.set_title('Top 20 classes par nombre d\'images (train)', fontsize=13)
ax.set_ylabel('Nombre d\'images')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Prétraitement & Data Augmentation

In [ ]:
IMG_SIZE   = 100   # Taille native Fruits-360
BATCH_SIZE = 64
EPOCHS_SCRATCH = 20
EPOCHS_TL      = 15

# Data augmentation pour le CNN from scratch
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.1
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=42
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=42
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f'Train : {train_gen.samples} images')
print(f'Val   : {val_gen.samples} images')
print(f'Test  : {test_gen.samples} images')

In [ ]:
# Visualisation de l'augmentation
aug_gen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

sample_class = classes[0]
sample_img_path = os.path.join(TRAIN_DIR, sample_class, os.listdir(os.path.join(TRAIN_DIR, sample_class))[0])
img_orig = plt.imread(sample_img_path)
img_arr  = img_orig.reshape((1,) + img_orig.shape)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle(f'Data Augmentation — {sample_class}', fontsize=13, fontweight='bold')
axes[0, 0].imshow(img_orig)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

i = 1
for batch in aug_gen.flow(img_arr, batch_size=1, seed=42):
    ax = axes.flat[i]
    ax.imshow(batch[0].astype('uint8'))
    ax.set_title(f'Aug #{i}')
    ax.axis('off')
    i += 1
    if i >= 10:
        break

plt.tight_layout()
plt.savefig('augmentation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Modèle 1 — CNN From Scratch

In [ ]:
def build_cnn_scratch(num_classes, img_size=100):
    """CNN simple construit from scratch."""
    model = models.Sequential([
        # Bloc 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same',
                      input_shape=(img_size, img_size, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Bloc 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Bloc 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # Classificateur
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='CNN_Scratch')
    return model

model_scratch = build_cnn_scratch(NUM_CLASSES)
model_scratch.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_scratch.summary()

In [ ]:
callbacks_scratch = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

print('Entraînement CNN From Scratch...')
t0 = time.time()

history_scratch = model_scratch.fit(
    train_gen,
    epochs=EPOCHS_SCRATCH,
    validation_data=val_gen,
    callbacks=callbacks_scratch,
    verbose=1
)

time_scratch = time.time() - t0
print(f'\n✅ Entraînement terminé en {time_scratch/60:.1f} minutes')

## 6. Modèle 2 — Transfer Learning (EfficientNetB0)

In [ ]:
def build_transfer_learning(num_classes, img_size=100):
    """EfficientNetB0 pré-entraîné sur ImageNet + tête de classification."""
    # Base pré-entraînée, sans la tête de classification
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(img_size, img_size, 3)
    )
    # Freeze complet dans un premier temps
    base_model.trainable = False

    # Nouvelle tête de classification
    inputs  = keras.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='TransferLearning_EfficientNetB0')
    return model, base_model

model_tl, base_model = build_transfer_learning(NUM_CLASSES)
model_tl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model_tl.summary()

In [ ]:
callbacks_tl = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

print('Phase 1 : Entraînement de la tête (base gelée)...')
t0 = time.time()

history_tl_phase1 = model_tl.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks_tl,
    verbose=1
)

print('Phase 2 : Fine-tuning (dégel des 50 dernières couches)...')
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model_tl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # LR réduit pour fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_tl_phase2 = model_tl.fit(
    train_gen,
    epochs=EPOCHS_TL,
    validation_data=val_gen,
    callbacks=callbacks_tl,
    verbose=1
)

time_tl = time.time() - t0
print(f'\n✅ Entraînement terminé en {time_tl/60:.1f} minutes')

# Fusion des historiques
for key in history_tl_phase1.history:
    history_tl_phase1.history[key] += history_tl_phase2.history[key]
history_tl = history_tl_phase1

## 7. Comparaison des deux modèles

In [ ]:
def plot_training_comparison(hist_scratch, hist_tl):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Comparaison CNN Scratch vs Transfer Learning', fontsize=15, fontweight='bold')

    # Accuracy
    axes[0].plot(hist_scratch.history['accuracy'],     label='CNN Scratch - Train',  color='#2196F3')
    axes[0].plot(hist_scratch.history['val_accuracy'], label='CNN Scratch - Val',    color='#2196F3', linestyle='--')
    axes[0].plot(hist_tl.history['accuracy'],          label='Transfer Learning - Train', color='#FF5722')
    axes[0].plot(hist_tl.history['val_accuracy'],      label='Transfer Learning - Val',   color='#FF5722', linestyle='--')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss
    axes[1].plot(hist_scratch.history['loss'],     label='CNN Scratch - Train',  color='#2196F3')
    axes[1].plot(hist_scratch.history['val_loss'], label='CNN Scratch - Val',    color='#2196F3', linestyle='--')
    axes[1].plot(hist_tl.history['loss'],          label='Transfer Learning - Train', color='#FF5722')
    axes[1].plot(hist_tl.history['val_loss'],      label='Transfer Learning - Val',   color='#FF5722', linestyle='--')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_comparison(history_scratch, history_tl)

In [ ]:
# Évaluation sur le jeu de test
print('Évaluation CNN From Scratch...')
loss_scratch, acc_scratch = model_scratch.evaluate(test_gen, verbose=0)

print('Évaluation Transfer Learning...')
loss_tl, acc_tl = model_tl.evaluate(test_gen, verbose=0)

# Nombre de paramètres
params_scratch = model_scratch.count_params()
params_tl      = model_tl.count_params()

print('\n' + '='*55)
print(f'{"":20} {"CNN Scratch":>15} {"Transfer Learning":>15}')
print('='*55)
print(f'{"Test Accuracy":20} {acc_scratch*100:>14.2f}% {acc_tl*100:>14.2f}%')
print(f'{"Test Loss":20} {loss_scratch:>15.4f} {loss_tl:>15.4f}')
print(f'{"Paramètres":20} {params_scratch:>15,} {params_tl:>15,}')
print(f'{"Temps (min)":20} {time_scratch/60:>15.1f} {time_tl/60:>15.1f}')
print('='*55)

In [ ]:
# Matrices de confusion (top 30 classes pour lisibilité)
def plot_confusion_matrix(model, generator, title, top_n=30):
    generator.reset()
    y_pred = model.predict(generator, verbose=0)
    y_pred_labels = np.argmax(y_pred, axis=1)
    y_true_labels = generator.classes

    # Sélection des top_n classes les plus fréquentes
    class_names = list(generator.class_indices.keys())
    top_indices = np.argsort(np.bincount(y_true_labels))[-top_n:]
    top_names   = [class_names[i] for i in top_indices]

    mask = np.isin(y_true_labels, top_indices)
    y_t  = y_true_labels[mask]
    y_p  = y_pred_labels[mask]

    # Remapping
    remap = {old: new for new, old in enumerate(top_indices)}
    y_t = np.array([remap[v] for v in y_t])
    y_p = np.array([remap.get(v, -1) for v in y_p])
    mask2 = y_p != -1
    y_t, y_p = y_t[mask2], y_p[mask2]

    cm = confusion_matrix(y_t, y_p, labels=range(top_n))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(16, 14))
    sns.heatmap(cm_norm, annot=False, fmt='.1%', cmap='Blues',
                xticklabels=top_names, yticklabels=top_names, ax=ax)
    ax.set_title(f'Matrice de Confusion — {title} (Top {top_n} classes)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Réel')
    ax.set_xlabel('Prédit')
    plt.xticks(rotation=45, ha='right', fontsize=7)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    fname = f'confusion_{title.replace(" ", "_").lower()}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    return y_true_labels, y_pred_labels

y_true, y_pred_scratch = plot_confusion_matrix(model_scratch, test_gen, 'CNN Scratch')
y_true, y_pred_tl      = plot_confusion_matrix(model_tl,      test_gen, 'Transfer Learning')

## 8. Visualisation GradCAM

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    """Génère une heatmap GradCAM pour une image donnée."""
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_output, preds = grad_model(img_array)
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_output = last_conv_output[0]
    heatmap = last_conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index), float(tf.reduce_max(preds))

def overlay_gradcam(img, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    return np.uint8(img * (1 - alpha) + heatmap_colored * alpha)

# Trouver la dernière couche Conv pour chaque modèle
def get_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, layers.Conv2D):
            return layer.name
    return None

last_conv_scratch = get_last_conv_layer(model_scratch)
print('Dernière conv CNN Scratch:', last_conv_scratch)

# Pour EfficientNet, cibler la dernière conv interne
last_conv_tl = 'top_conv'  # Couche top de EfficientNetB0
print('Dernière conv Transfer Learning:', last_conv_tl)

In [ ]:
# Visualisation GradCAM sur 6 exemples
selected_classes_gradcam = classes[:6]
fig, axes = plt.subplots(6, 4, figsize=(16, 22))
fig.suptitle('GradCAM — Ce que chaque modèle "regarde"', fontsize=15, fontweight='bold')

headers = ['Image originale', 'GradCAM CNN Scratch', 'GradCAM Transfer Learning', 'Comparaison']
for ax, h in zip(axes[0], headers):
    ax.set_title(h, fontsize=10, fontweight='bold')

class_names_inv = {v: k for k, v in test_gen.class_indices.items()}

for row, cls in enumerate(selected_classes_gradcam):
    img_path = os.path.join(TRAIN_DIR, cls, os.listdir(os.path.join(TRAIN_DIR, cls))[5])
    img_orig = plt.imread(img_path)
    img_orig = cv2.resize(img_orig, (IMG_SIZE, IMG_SIZE))
    img_arr  = np.expand_dims(img_orig.astype('float32') / 255.0, axis=0)

    # GradCAM CNN Scratch
    try:
        hm_scratch, pred_s, conf_s = make_gradcam_heatmap(img_arr, model_scratch, last_conv_scratch)
        overlay_s = overlay_gradcam(img_orig, hm_scratch)
    except Exception:
        overlay_s = img_orig
        conf_s = 0.0

    # GradCAM Transfer Learning
    try:
        hm_tl, pred_tl, conf_tl = make_gradcam_heatmap(img_arr, model_tl, last_conv_tl)
        overlay_tl = overlay_gradcam(img_orig, hm_tl)
    except Exception:
        overlay_tl = img_orig
        conf_tl = 0.0

    axes[row, 0].imshow(img_orig)
    axes[row, 0].set_ylabel(cls, fontsize=8, rotation=0, labelpad=60)

    axes[row, 1].imshow(overlay_s)
    axes[row, 1].set_title(f'Conf: {conf_s:.1%}', fontsize=8)

    axes[row, 2].imshow(overlay_tl)
    axes[row, 2].set_title(f'Conf: {conf_tl:.1%}', fontsize=8)

    # Comparaison côte à côte
    combined = np.hstack([overlay_s, overlay_tl])
    axes[row, 3].imshow(combined)

    for ax in axes[row]:
        ax.axis('off')

plt.tight_layout()
plt.savefig('gradcam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Prédictions sur des images test

In [ ]:
# Prédictions côte à côte sur 12 images test
class_names_list = list(test_gen.class_indices.keys())

test_gen.reset()
batch_imgs, batch_labels = next(test_gen)

preds_scratch = model_scratch.predict(batch_imgs[:12], verbose=0)
preds_tl      = model_tl.predict(batch_imgs[:12], verbose=0)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Prédictions sur des images test', fontsize=15, fontweight='bold')

for i, ax in enumerate(axes.flat):
    img = batch_imgs[i]
    true_label = class_names_list[np.argmax(batch_labels[i])]

    pred_s_label = class_names_list[np.argmax(preds_scratch[i])]
    pred_tl_label = class_names_list[np.argmax(preds_tl[i])]
    conf_s  = np.max(preds_scratch[i])
    conf_tl = np.max(preds_tl[i])

    ax.imshow(img)
    correct_s  = '✅' if pred_s_label  == true_label else '❌'
    correct_tl = '✅' if pred_tl_label == true_label else '❌'
    ax.set_title(
        f'Vrai: {true_label}\n'
        f'Scratch {correct_s}: {pred_s_label[:12]} ({conf_s:.0%})\n'
        f'TL {correct_tl}: {pred_tl_label[:12]} ({conf_tl:.0%})',
        fontsize=7
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('predictions_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Bilan & Conclusion

In [ ]:
print('='*65)
print('                    BILAN COMPARATIF FINAL')
print('='*65)
print(f'  Métrique            CNN From Scratch    Transfer Learning')
print('-'*65)
print(f'  Test Accuracy       {acc_scratch*100:>12.2f}%    {acc_tl*100:>12.2f}%')
print(f'  Test Loss           {loss_scratch:>16.4f}    {loss_tl:>12.4f}')
print(f'  Nb Paramètres       {params_scratch:>16,}    {params_tl:>12,}')
print(f'  Temps (min)         {time_scratch/60:>16.1f}    {time_tl/60:>12.1f}')
print('='*65)

print()
print('Conclusion :')
if acc_tl > acc_scratch:
    gain = (acc_tl - acc_scratch) * 100
    print(f'  → Le Transfer Learning surpasse le CNN Scratch de +{gain:.1f}%')
    print(f'  → Malgré {params_tl/params_scratch:.1f}x plus de paramètres, il converge plus vite')
    print(f'  → Les features ImageNet sont transférables aux fruits !')
else:
    print(f'  → Les deux modèles sont comparables sur ce dataset')
    print(f'  → Fruits-360 (fond blanc uniforme) favorise les CNNs simples')

In [ ]:
# Sauvegarde des modèles
model_scratch.save('cnn_scratch_fruits360.h5')
model_tl.save('transfer_learning_fruits360.h5')
print('✅ Modèles sauvegardés')

# Téléchargement des résultats
from google.colab import files
for f in ['training_comparison.png', 'confusion_cnn_scratch.png',
          'confusion_transfer_learning.png', 'gradcam_comparison.png',
          'predictions_comparison.png', 'dataset_samples.png']:
    if os.path.exists(f):
        files.download(f)
print('✅ Figures téléchargées')